In [2]:
import os
from dotenv import load_dotenv
from typing import List, Optional, TypedDict

from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.prompts import load_prompt
from langchain_groq import ChatGroq
from langchain_community.graphs import Neo4jGraph
from langgraph.graph import StateGraph, END

# --- 1. CONFIGURATION & INITIALIZATION (from your notebook) ---
# Assuming variables are already loaded from .env as per your notebook

load_dotenv()

NEO4J_URL = os.getenv("NEO4J_URL")
NEO4J_USER = os.getenv("NEO4J_USER")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
groq_api_key = os.getenv("GROQ_API_KEY")

graph = Neo4jGraph(
    url=NEO4J_URL, 
    username=NEO4J_USER, 
    password=NEO4J_PASSWORD, 
    database="78ce8520",
    refresh_schema=True 
)

llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0)
cypher_prompt = load_prompt("cypher-few-shot.yaml")
qa_prompt_template = load_prompt("qa-prompt.yaml")


In [5]:

# --- 2. STATE DEFINITION ---
class AgentState(TypedDict):
    question: str
    cypher: Optional[str]
    error: Optional[str]
    results: List
    iterations: int
    final_response: Optional[str]
    in_scope: bool

# --- 3. NODE DEFINITIONS ---
SCOPE_SYSTEM_PROMPT = """You are an expert on global crude oil supply chain data, imports, and trade volumes.
If a question is NOT about oil trade or energy logistics, you must flag it as out of scope."""

def check_scope_node(state: AgentState):
    """Initializes the state and checks project scope."""
    check = llm.invoke([
        SystemMessage(content=SCOPE_SYSTEM_PROMPT),
        HumanMessage(content=f"Is this question about oil trade or energy logistics? Answer 'YES' or 'NO' only: {state['question']}")
    ])
    
    in_scope = "YES" in check.content.upper()
    
    return {
        "in_scope": in_scope, 
        "iterations": 0, 
        "results": [], 
        "error": None, 
        "cypher": None,
        "final_response": None
    }

def generate_cypher_node(state: AgentState):
    """Generates Cypher using your FewShotPromptTemplate."""
    error_msg = f"\n\nPrevious attempt failed with error: {state['error']}. Fix the query." if state['error'] else ""
    
    # Use the schema and question to format your YAML-loaded prompt
    formatted_prompt = cypher_prompt.format(
        question=state["question"], 
        schema=graph.schema
    ) + error_msg
    
    response = llm.invoke(formatted_prompt)
    return {"cypher": response.content, "iterations": state["iterations"] + 1}

def execute_cypher_node(state: AgentState):
    """Attempts execution on Neo4j."""
    try:
        # Clean the Cypher string (remove markdown if the LLM added it)
        clean_cypher = state["cypher"].replace("```cypher", "").replace("```", "").strip()
        results = graph.query(clean_cypher)
        return {"results": results, "error": None}
    except Exception as e:
        return {"error": str(e), "results": []}

def responder_node(state: AgentState):
    """Generates the final natural language answer or fallback."""
    # 1. Fallback for Out-of-Scope
    if not state["in_scope"]:
        return {"final_response": "I am sorry, but your request is outside the objectives of this oil trade intelligence project. I can only assist with queries related to crude oil imports and global energy logistics."}

    # 2. Fallback for Repeated Cypher Errors (Humble Refusal)
    if state["error"] and state["iterations"] >= 2:
        return {"final_response": "I cannot answer this query right now due to technical difficulties with the database query."}

    # 3. Handle Empty Results
    if not state["results"]:
        return {"final_response": "No matching oil trade data was found for your specific query records."}

    # 4. Standard Response using your YAML QA Prompt logic
    # We use the QA prompt template loaded from your YAML to maintain formatting rules
    full_qa_prompt = qa_prompt_template.format(
        question=state["question"],
        context=str(state["results"])
    )
    
    summary = llm.invoke(full_qa_prompt)
    return {"final_response": summary.content}



In [9]:

# --- 4. GRAPH CONSTRUCTION ---
workflow = StateGraph(AgentState)

workflow.add_node("check_scope", check_scope_node)
workflow.add_node("generate_cypher", generate_cypher_node)
workflow.add_node("execute_cypher", execute_cypher_node)
workflow.add_node("responder", responder_node)

workflow.set_entry_point("check_scope")

# Conditional: Check scope first
workflow.add_conditional_edges(
    "check_scope", 
    lambda x: "generate_cypher" if x["in_scope"] else "responder"
)

workflow.add_edge("generate_cypher", "execute_cypher")

# Conditional: Self-Correction (Retry logic)
def retry_logic(state):
    if state["error"] and state["iterations"] < 2:
        return "generate_cypher"
    return "responder"

workflow.add_conditional_edges("execute_cypher", retry_logic)
workflow.add_edge("responder", END)

# Compile the app
app = workflow.compile()

# --- 5. EXECUTION WRAPPER ---
def ask_sentinel(question: str):
    inputs = {"question": question}
    final_state = app.invoke(inputs)
    return final_state["final_response"]


In [10]:
print(ask_sentinel("Largest exporter of crude oil in South East Asia?"))

Based on total trade for the latest year, the largest exporter of crude oil in South East Asia is Malaysia with 595.5 million barrels.


In [11]:
res = ask_sentinel("Which country provided the most barrels to the USA in 202401?")
print(res)

Based on total trade for Jan 2024, Canada provided the most barrels to the USA, delivering 126.42 million barrels.


In [12]:
res = ask_sentinel("How much barrels India imported in 202305?")
print(res)

Based on total trade for May 2023, India imported 161.24 million barrels.


In [13]:
res = ask_sentinel("If Russians exports drop by 20%, which partner countries have the highest risk based on their number of barrels")
print(res)

Based on total trade for the latest year, the partner countries at highest risk due to a 20% drop in Russian exports are:  
China – potential loss of 157.93 million barrels  
India – potential loss of 133.96 million barrels  
Hungary – potential loss of 7.01 million barrels  
Slovakia – potential loss of 6.18 million barrels  
Czechia – potential loss of 3.95 million barrels


In [14]:
res = ask_sentinel("If Russians exports drop by 20%, which partner countries have the highest risk based on their historical reliance?")
print(res)

Based on total trade for 2024, the partner countries with the highest risk due to a 20% drop in Russian exports are:

Armenia – reliance: 100.0% – projected loss: 0.0 million barrels  
Georgia – reliance: 100.0% – projected loss: 0.02 million barrels  
Azerbaijan – reliance: 89.44% – projected loss: 2.44 million barrels  
Slovakia – reliance: 87.85% – projected loss: 6.18 million barrels  
Hungary – reliance: 86.21% – projected loss: 7.01 million barrels


In [15]:
res = ask_sentinel("Which country is most reliable on Russia for their imports?")
print(res)

Based on total trade for 2023, Georgia is the most reliable on Russia for its imports, with a reliance of 100.0 % and an import volume of 0.08 million barrels.


In [26]:
res = ask_sentinel("Which country is most reliable on Russia for their imports for 2027?")
print(res)

No matching oil trade data was found for your specific query records.


In [28]:
res = ask_sentinel("Find instances where an importer is importing the same commodity from two different exporter at a price difference of >30%.")
print(res)

I am sorry, but your request is outside the objectives of this oil trade intelligence project. I can only assist with queries related to crude oil imports and global energy logistics.


In [27]:
res = ask_sentinel("Who is the largest exporter of oil?")
print(res)

Based on total trade for 2023, the largest exporter of oil is Saudi Arabia with 2145.19 million barrels.


In [19]:
res = ask_sentinel("If there is a war in the middle east then which countries imports will be affected and by how much?\
Will India be affected?")
print(res)

Based on total trade for the latest year, the following importing countries would see their oil imports affected by a war in the Middle East:

China – 1,789.72 million barrels  
Japan – 817.61 million barrels  
India – 809.27 million barrels  
Rep. of Korea – 728.17 million barrels  
Thailand – 236.63 million barrels  

India is indeed among the affected importers, with an estimated import volume of 809.27 million barrels that could be impacted.


In [20]:
res = ask_sentinel("Largest importer of crude oil in South East Asia?")
print(res)

Based on total trade for the reported period, the largest importer of crude oil in South East Asia is Thailand with 402.75 million barrels.


In [21]:
res = ask_sentinel("Largest exporter of crude oil in South East Asia?")
print(res)

Based on total trade for the most recent year, the largest exporter of crude oil in South East Asia is Malaysia with 595.5 million barrels.


In [22]:
res = ask_sentinel("How to bake French bread?")
print(res)

I am sorry, but your request is outside the objectives of this oil trade intelligence project. I can only assist with queries related to crude oil imports and global energy logistics.


In [23]:
res = ask_sentinel("If there is a conflict between Iran and Israel then which countries imports will be affected?")
print(res)

#The LLM does not inherently know that a conflict between Iran and Israel might lead to a blockade of the Strait of Hormuz,
#  which would affect all Middle Eastern oil (Saudi, UAE, Kuwait, etc.). 
#It only knows what is in the relationship lines.


# 10. GEOPOLITICAL IMPACT: If a question mentions a conflict or 'risk' in the Middle East (e.g., Iran, Israel, Red Sea):
        # a) First, query direct trade for those specific countries.
        # b) Second, automatically query the total imports from the broader region: 
        #    ['Saudi Arabia', 'United Arab Emirates', 'Iraq', 'Kuwait', 'Qatar', 'Oman'].
        # c) Return BOTH sets of data so the user can see the direct and regional exposure.

Based on total trade for the reported period, the imports of the Netherlands (3.94 million barrels) would be affected.
